# SpAM Pilot Analysis

Descriptive analysis of pilot data collected via the SpAM (Spatial Arrangement Method) task.
Data directory: `data/pilot/` (local only, gitignored).

In [ ]:
import warnings
import plotly.io as pio

from analysis.pilot.parser import load_pilot_data
from analysis.pilot.figures import (
    fig_completion_status,
    fig_trial_duration_per_subject,
    fig_moves_per_subject,
    fig_duration_progression,
    fig_moves_progression,
    fig_duration_vs_moves,
    fig_within_subject_variability,
    fig_demographics,
    fig_pairwise_distance_distribution,
)

pio.renderers.default = "browser"

## 0. Load data

In [ ]:
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    data = load_pilot_data("data/pilot")

df_trials = data["trials"]
df_status = data["status"]

for w in caught_warnings:
    print(f"[{w.category.__name__}] {w.message}")

print(f"\nTrials dataframe: {df_trials.shape[0]} rows × {df_trials.shape[1]} cols")
print(df_status["completion_status"].value_counts().to_string())

## 1. Completion status

In [ ]:
fig_completion_status(df_status).show()

## 2. Trial duration per subject

In [ ]:
fig_trial_duration_per_subject(df_trials).show()

## 3. Number of moves per subject

In [ ]:
fig_moves_per_subject(df_trials).show()

## 4. Trial duration over task progression

In [ ]:
fig_duration_progression(df_trials).show()

## 5. Moves over task progression

In [ ]:
fig_moves_progression(df_trials).show()

## 6. Trial duration vs. number of moves

In [ ]:
fig_duration_vs_moves(df_trials).show()

## 7. Within-subject variability and reliability

In [ ]:
fig_within_subject_variability(df_trials).show()

## 8. Participant demographics

In [ ]:
fig_demographics(df_trials).show()

## 9. Pairwise distance distribution

In [ ]:
fig_pairwise_distance_distribution(df_trials).show()

## Summary statistics

In [ ]:
summary = (
    df_trials
    .assign(
        rt_s=df_trials["rt"] / 1000,
        qc_flag_int=df_trials["qc_flag"].astype(int),
    )
    .groupby("task_version")
    .agg(
        n_subjects=("participant_id",  "nunique"),
        n_trials=("trial_number",      "count"),
        rt_s_mean=("rt_s",             "mean"),
        rt_s_median=("rt_s",           "median"),
        rt_s_sd=("rt_s",               "std"),
        rt_s_min=("rt_s",              "min"),
        rt_s_max=("rt_s",              "max"),
        moves_mean=("n_moves",         "mean"),
        moves_median=("n_moves",       "median"),
        moves_sd=("n_moves",           "std"),
        qc_flag_rate=("qc_flag_int",   "mean"),
    )
    .T
    .rename(columns=lambda v: f"v{v:g}")
    .round(2)
)
summary